<a href="https://colab.research.google.com/github/LinaMariaCastro/curso-ia-para-economia/blob/main/clases/4_Aprendizaje_no_supervisado/2_Taller_Apriori.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# **Inteligencia Artificial con Aplicaciones en Economía I**

- 👩‍🏫 **Profesora:** [Lina María Castro](https://www.linkedin.com/in/lina-maria-castro)  
- 📧 **Email:** [lmcastroco@gmail.com](mailto:lmcastroco@gmail.com)  
- 🎓 **Universidad:** Universidad Externado de Colombia - Facultad de Economía

# **Taller: Análisis de Patrones de Consumo Internacional con Apriori**

**IMPORTANTE**: Guarda una copia de este notebook en tu Google Drive o computador.

**Taller en grupos de 3**

**Nombres estudiantes:**

- Juan Esteban Barrantes
- Daniel Caicedo
- Juan Camilo Ordoñez

**Forma de entrega:**

- Nombrar el archivo de la siguiente forma:“Taller_Apriori_apellidos.ipynb”.
- Suba el Jupyter Notebook a su cuenta en Github y envíe el link en el siguiente Forms: https://forms.cloud.microsoft/r/qERdEpXpmx.

**IMPORTANTE:** No se recibirán talleres en Google Colab, el notebook debe estar subido en Github.

**Plazo de entrega:**

21 de abril de 2026, máximo a las 11:59 p.m. Tenga en cuenta que luego de esa hora el formulario en forms se cierra. El Jupupyter Notebook también debe quedar subido en Github antes de esa hora.

**Instrucciones Generales:**

Completa el código en las celdas marcadas con `### TU CÓDIGO AQUÍ ###`. Puedes añadir más celdas si lo requieres.

**Caso de Estudio: Consultoría para Global Retail Inc.**

**Contexto:** Una firma multinacional de e-commerce, "Global Retail Inc.", te ha contratado como consultor de datos. La empresa opera en múltiples países y ha notado que sus ventas y la efectividad de sus campañas de marketing varían significativamente entre regiones. Su hipótesis es que los patrones de compra y las asociaciones de productos son diferentes en cada mercado.

**Tu Misión:** Analizar el historial de transacciones de la empresa para descubrir y comparar las reglas de asociación de productos para dos de sus mercados más importantes en Latinoamérica: México y Colombia. Tu objetivo final es entregar recomendaciones de negocio accionables (ej. estrategias de cross-selling, promociones personalizadas) basadas en los patrones de consumo que descubras en cada país.

**Dataset:** Encuentra mayor información en el archivo "diccionario_alimentos_retail_top30.xlsx".

## Ejercicio 1: Configuración Inicial, Carga y Exploración de Datos

1.1 Importa las librerías necesarias

In [ ]:
!pip install mlxtend

In [ ]:
import os
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Configuraciones de visualización
pd.options.display.max_columns = None
pd.options.display.float_format = '{:,.2f}'.format

1.2 Carga el dataset "alimentos_retail_top30.csv" que se encuentra en el repositorio del curso, carpeta "datasets". El dataframe debe llamarse "df".

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path = '/content/drive/MyDrive/Dataset/4_Aprendizaje_no_supervisado'
# Para establecer el directorio de los archivos
os.chdir(path)

In [ ]:
### TU CÓDIGO AQUÍ ###
df = pd.read_csv('alimentos_retail_top30.csv')

In [ ]:
# Debe ser (6899, 8)
print("Dimensiones del DataFrame:")
print(df.shape)

Dimensiones del DataFrame:
(6899, 8)


In [ ]:
print("\nInformación general del DataFrame:")
df.info()


Información general del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6899 entries, 0 to 6898
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   InvoiceNo    6899 non-null   object 
 1   StockCode    6899 non-null   int64  
 2   Description  6899 non-null   object 
 3   Quantity     6899 non-null   int64  
 4   InvoiceDate  6899 non-null   object 
 5   UnitPrice    6899 non-null   float64
 6   CustomerID   6879 non-null   float64
 7   Country      6899 non-null   object 
dtypes: float64(2), int64(2), object(4)
memory usage: 431.3+ KB


In [ ]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536000,94537,HARINA DE MAÍZ,5,2023-01-07 01:09:00,2.76,"17,452.00",Colombia
1,536000,87297,QUESO MUZZARELLA,2,2023-01-07 01:09:00,4.69,"17,779.00",Colombia
2,536001,94537,HARINA DE MAÍZ,4,2023-01-07 11:51:00,2.76,"14,933.00",Colombia
3,536001,87297,QUESO MUZZARELLA,3,2023-01-07 11:51:00,4.69,"14,957.00",Colombia
4,536002,26907,CAFÉ,4,2023-01-02 01:54:00,2.36,"15,202.00",Colombia


1.3 Revisa si hay valores nulos en alguna columna y cuántos son

In [ ]:
df.isna().sum()

,0
InvoiceNo,0
StockCode,0
Description,0
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,20
Country,0


1.4 Genera las estadísticas descriptivas de las variables numéricas

In [ ]:
df.describe()

,StockCode,Quantity,UnitPrice,CustomerID
count,"6,899.00","6,899.00","6,899.00","6,879.00"
mean,"55,544.94",3.00,3.42,"15,024.12"
std,"25,875.73",1.43,1.06,"1,732.95"
min,"26,907.00",-5.00,1.65,"12,000.00"
25%,"31,048.00",2.00,2.36,"13,524.00"
50%,"42,889.00",3.00,3.39,"15,041.00"
75%,"87,297.00",4.00,4.44,"16,530.50"
max,"95,931.00",5.00,4.90,"17,999.00"


1.5 Observando las salidas del ejercicio anterior, ¿qué problemas potenciales identificas en las columnas CustomerID y Quantity?

En la columna CustomerID el count (6,879) es menor que el de las otras columnas (6,899), por lo que hay valores faltantes (missing values).
En la columna Quantity el valor mínimo es -5, lo cual no tiene sentido en condiciones normales (no se puede vender “-5” unidades).
Esto suele indicar devoluciones (returns) registradas como cantidades negativas o posibles errores en los datos.

## Ejercicio 2: Limpieza y Preprocesamiento de Datos

Los datos del mundo real rara vez son perfectos. Antes de cualquier análisis, debemos "sanear" nuestro dataset. Completa el código en cada paso según las instrucciones.

Crea un nuevo dataframe llamado "df_limpio" para los siguientes puntos.

2.1 **Manejo de Valores Nulos**: Las transacciones sin un CustomerID no son útiles para nosotros, ya que no podemos agrupar las compras de un cliente específico.

In [ ]:
# TAREA: Elimina todas las filas donde 'CustomerID' es nulo.
### TU CÓDIGO AQUÍ ###
df_limpio =df.dropna()

In [ ]:
# El tipo de dato de CustomerID debe ser entero
### TU CÓDIGO AQUÍ ###
df_limpio['CustomerID'] = df_limpio['CustomerID'].astype('int')

2.2 **Limpieza de Descripciones de Productos** Las descripciones pueden tener espacios en blanco al inicio o al final que podrían hacer que un mismo producto se cuente como dos diferentes.

In [ ]:
# TAREA: # Verifica cuántas descripciones únicas hay.
descripciones_unicas = df_limpio['Description'].nunique()
print(f"Total de descripciones únicas: {descripciones_unicas}")

Total de descripciones únicas: 25


In [ ]:
# TAREA: Limpia la columna 'Description' eliminando espacios extra al inicio y al final.
df_limpio['Description'] = df_limpio['Description'].str.strip()

In [ ]:
# TAREA: Verifica cuántas descripciones únicas quedaron después de la limpieza.
descripciones_unicas = df_limpio['Description'].nunique()
print(f"Total de descripciones únicas: {descripciones_unicas}")

Total de descripciones únicas: 20


2.3 **Filtrado de Transacciones Anómalas**: Las facturas (InvoiceNo) que empiezan con 'C' indican una cancelación. Estas no son compras reales y deben ser eliminadas. Del mismo modo, las cantidades (Quantity) negativas representan devoluciones.

In [ ]:
# TAREA: Elimina las filas que correspondan a cancelaciones.
df_limpio = df_limpio[(df_limpio['InvoiceNo'].str.startswith('C') != True) & (df_limpio['Quantity'] > 0)]

In [ ]:
# TAREA: Elimina las filas con cantidades negativas.
### TU CÓDIGO AQUÍ ###


In [ ]:
# Verifiquemos las dimensiones del DataFrame después de la limpieza. Debe ser (6864, 8)
df_limpio.shape

(6864, 8)

## Ejercicio 3: Análisis Comparativo por País

Ahora que los datos están limpios, vamos a segmentarlos y a aplicar el algoritmo Apriori para encontrar los patrones de compra en México y Colombia.

**Preparación de la Cesta de Mercado (Función)**

La siguiente función toma un dataframe, lo agrupa por factura y descripción, y lo transforma en el formato de matriz binaria que necesita el algoritmo Apriori. Estudia esta función, no necesitas modificarla.

In [ ]:
def preparar_cesta(dataframe, pais):
    """Filtra por país y prepara la matriz de transacciones."""

    # Filtrar por el país de interés
    df_pais = dataframe[dataframe['Country'] == pais]

    # Crear la cesta: agrupar productos por factura
    cesta = (df_pais.groupby(['InvoiceNo', 'Description'])['Quantity']
             .sum().unstack().reset_index().fillna(0)
             .set_index('InvoiceNo'))

    # Convertir todas las cantidades positivas a 1 y todo lo demás a 0
    cesta_encoded = (cesta > 0).astype(int)

    return cesta_encoded

3.1 Análisis para México

In [ ]:
# TAREA: Usa la función preparar_cesta para obtener la matriz de transacciones de México. Almacena el resultado en la variable cesta_mx.
cesta_mx = preparar_cesta(df_limpio, "México")

In [ ]:
cesta_mx

Description,AGUACATE,CEBOLLA,CHILE JALAPEÑO,CILANTRO,FRIJOL NEGRO,LIMÓN,QUESO FRESCO,TOMATE,TORTILLAS DE MAÍZ,TOTOPOS
InvoiceNo,,,,,,,,,,
537000,0,0,0,0,1,0,0,0,1,0
537001,0,1,1,1,0,0,0,1,0,0
537002,0,0,0,0,1,0,0,1,1,0
537003,0,1,1,1,0,0,0,1,0,0
537004,0,0,0,1,0,0,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...
537995,0,0,0,0,1,0,0,0,1,0
537996,0,1,1,1,0,0,0,1,0,0
537997,0,0,0,0,0,0,0,1,0,1


In [ ]:
# TAREA: Aplica el algoritmo apriori para encontrar itemsets con un soporte mínimo de 2%.
# Almacena el resultado en la variable frequent_itemsets_mx.
# Muestra los 10 itemsets con el soporte más alto.
frequent_itemsets_mx =apriori(cesta_mx, min_support=0.02, use_colnames=True)
frequent_itemsets_mx.sort_values(by='support', ascending=False).head(10)

,support,itemsets
2,0.42,(CHILE JALAPEÑO)
7,0.41,(TOMATE)
3,0.41,(CILANTRO)
1,0.41,(CEBOLLA)
8,0.38,(TORTILLAS DE MAÍZ)
4,0.36,(FRIJOL NEGRO)
0,0.35,(AGUACATE)
5,0.35,(LIMÓN)
9,0.33,(TOTOPOS)
31,0.33,"(CHILE JALAPEÑO, TOMATE)"


In [ ]:
# TAREA: Genera las reglas de asociación. Queremos reglas con un Lift mayor a 2. Almacena el resultado en la variable rules_mx.
rules_mx = association_rules(frequent_itemsets_mx, metric="lift", min_threshold=1.2)


In [ ]:
# Ordena las reglas por Lift y Confianza de mayor a menor, muestra solamente las primeras 10 filas y las siguientes columnas:
# 'antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift'
rules_mx = rules_mx[['antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift']]
rules_mx.sort_values(['lift', 'confidence'], ascending=[False, False]).head(15)

,antecedents,consequents,antecedent support,consequent support,confidence,lift
236,"(CHILE JALAPEÑO, CEBOLLA)","(TOMATE, CILANTRO)",0.32,0.32,0.92,2.90
237,"(TOMATE, CILANTRO)","(CHILE JALAPEÑO, CEBOLLA)",0.32,0.32,0.93,2.90
239,"(CILANTRO, CEBOLLA)","(CHILE JALAPEÑO, TOMATE)",0.31,0.33,0.95,2.88
234,"(CHILE JALAPEÑO, TOMATE)","(CILANTRO, CEBOLLA)",0.33,0.31,0.90,2.88
59,"(AGUACATE, LIMÓN)",(TOTOPOS),0.27,0.33,0.93,2.82
62,(TOTOPOS),"(AGUACATE, LIMÓN)",0.33,0.27,0.75,2.82
235,"(CHILE JALAPEÑO, CILANTRO)","(TOMATE, CEBOLLA)",0.32,0.33,0.92,2.81
238,"(TOMATE, CEBOLLA)","(CHILE JALAPEÑO, CILANTRO)",0.33,0.32,0.91,2.81
207,"(AGUACATE, TOMATE, LIMÓN)",(TOTOPOS),0.02,0.33,0.91,2.76
216,(TOTOPOS),"(AGUACATE, TOMATE, LIMÓN)",0.33,0.02,0.06,2.76


3.3 Observa las 3 reglas con el Lift más alto para México (1, 3 y 5). **Interprétalas:** ¿Qué te dicen estas asociaciones? ¿Qué tipo de productos son?

En la regla 1, vemos que son productos complementarios, no sustitutos, identificamos que representan un combo culinario tradicional.
En la regla 3, se refuerza que hay un cluster fuerte de ingredientes de cocina mexicana básica, se identifica una alta coherencia cultural en el consumo.
En la regla 5, no solo es lo que se va a cocinar, también consumo listo.
Mezcla de: ingrediente (aguacate, limón) y producto listo (totopos).

3.4 Para cada una de las 3 reglas (1, 3 y 5), interpreta el Soporte para el antecedente y el consecuente, la Confianza y el Lift

Para la regla 1:

Soporte antecedente (0.32) indica que el 32% de las transacciones incluyen
jalapeño + cebolla.
Soporte consecuente (0.32) indica que el 32% también compran tomate + cilantro.
Confianza (0.92) muestra que si compran jalapeño + cebolla el 92% compra tambein tomate + cilantro.
Lift (2.90) quiere decir que es 2.9 veces más probable que ocurra esta combinación que por azar.

Para la regla 3:

Soporte antecedente (~0.31) es bastante común también
Confianza (0.95) es aún más fuerte, casi todos los que compran cilantro + cebolla completan la receta
Lift (2.88) Asociación muy alta

Para la regla 5:

Soporte antecedente (0.27) el 27% compran aguacate + limón
Soporte consecuente (0.33) el 33% compran totopos
Confianza (0.93) muestra que si compran aguacate + limón  el 93% compra tambien totopos
Lift (2.82) Muy fuerte relación

3.5 **Recomendación de Negocio:** Basado en estas reglas, ¿qué promoción o estrategia de venta específica podrías sugerir para el mercado mexicano?

Combos listos

Crear paquetes tipo: “Kit Salsa Mexicana” con
tomate,
cebolla,
cilantro,
jalapeño. Esto reduce la fricción y aumenta ticket promedio.

3.6 Análisis para Colombia

In [ ]:
# TAREA: Usa la función preparar_cesta para obtener la matriz de transacciones de Colombia. Almacena el resultado en la variable cesta_co.
cesta_co = preparar_cesta(df_limpio, "Colombia")

In [ ]:
# TAREA: Aplica el algoritmo apriori con un soporte mínimo del 2%.
# Almacena el resultado en la variable frequent_itemsets_co.
# Muestra los 10 itemsets con el soporte más alto.
### TU CÓDIGO AQUÍ ###
frequent_itemsets_co =apriori(cesta_co, min_support=0.02, use_colnames=True)
frequent_itemsets_co.sort_values(by='support', ascending=False).head(10)

,support,itemsets
3,0.41,(CAFÉ)
4,0.41,(FRIJOL CARGAMANTO)
0,0.40,(ACEITE DE GIRASOL)
2,0.40,(AZÚCAR)
7,0.39,(LECHE)
1,0.38,(ARROZ)
9,0.35,(QUESO MUZZARELLA)
5,0.34,(HARINA DE MAÍZ)
37,0.32,"(CAFÉ, LECHE)"
31,0.32,"(AZÚCAR, LECHE)"


In [ ]:
# TAREA: Genera las reglas de asociación con un Lift mayor a 2. Almacena el resultado en la variable rules_co.
rules_co = association_rules(frequent_itemsets_co, metric="lift", min_threshold=1.2)

In [ ]:
# Ordena las reglas por Lift y Confianza de mayor a menor, muestra solamente las primeras 10 filas y las siguientes columnas:
# 'antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift'
rules_co = rules_co[['antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift']]
rules_co.sort_values(['lift', 'confidence'], ascending=[False, False]).head(15)

,antecedents,consequents,antecedent support,consequent support,confidence,lift
178,"(AZÚCAR, CAFÉ, FRIJOL CARGAMANTO)",(LECHE),0.05,0.39,0.96,2.44
189,(LECHE),"(AZÚCAR, CAFÉ, FRIJOL CARGAMANTO)",0.39,0.05,0.11,2.44
74,"(AZÚCAR, CAFÉ)",(LECHE),0.32,0.39,0.95,2.41
79,(LECHE),"(AZÚCAR, CAFÉ)",0.39,0.32,0.76,2.41
18,"(FRIJOL CARGAMANTO, ACEITE DE GIRASOL)",(ARROZ),0.30,0.38,0.91,2.41
19,(ARROZ),"(FRIJOL CARGAMANTO, ACEITE DE GIRASOL)",0.38,0.30,0.73,2.41
210,"(PAN TAJADO, CAFÉ, AZÚCAR)",(LECHE),0.03,0.39,0.94,2.39
221,(LECHE),"(PAN TAJADO, CAFÉ, AZÚCAR)",0.39,0.03,0.07,2.39
198,"(AZÚCAR, CAFÉ, HUEVOS)",(LECHE),0.04,0.39,0.92,2.36
209,(LECHE),"(AZÚCAR, CAFÉ, HUEVOS)",0.39,0.04,0.09,2.36


3.7 Observa las 3 reglas con el Lift más alto para Colombia (1, 3 y 5). **Interprétalas:** ¿Qué patrones de consumo específicos del mercado colombiano revelan estas reglas? ¿Son diferentes a las de México?

E la regla 1, el consumidor colombiano compra de forma integral (mercado completo), no por ocasión específica.
No es “receta puntual” como en México → es canasta básica del hogar.

En la regla 3, se ve el patrón fuerte de consumo de café con leche (tinto con leche / café con leche), es casi un hábito diario estructural.

En la regla 5, representa el clásico: arroz + frijol + aceite. Es un combo alimenticio tradicional, no un snack ni algo ocasional.

3.8 Para cada una de las 3 reglas (1, 3 y 5), interpreta el Soporte para el antecedente y el consecuente, la Confianza y el Lift

Regla 1

Soporte antecedente (0.05): Solo 5% compran esa combinación completa
Soporte consecuente (0.39): 39% compran leche (producto masivo)
Confianza (0.96): Si compran esos productos el 96% compran leche
Lift (2.44): Es mucho más probable que el promedio

Regla 3

Soporte antecedente (0.32): Es bastante común
Confianza (0.95): Casi todos los que compran café + azúcar → compran leche
Lift (2.41) Este sí es un patrón masivo y estable

Regla 5

Soporte antecedente (0.30): Muy frecuente
Confianza (0.91): Alta probabilidad de complementar con arroz
Lift (2.41): Esto es core de alimentación colombiana

3.9 **Recomendación de Negocio:** ¿Qué campaña de marketing (diferente a la de México) podrías diseñar para los clientes colombianos?

Campaña: “Arma tu mercado semanal”

Idea:

Ofrecer combos tipo: Desayuno colombiano "café + azúcar + leche" o almuerzo básico "arroz + frijol + aceite"

No vender productos, es mejor vender abastecimiento.